# Day 6 — Capstone: Deploy Your RAG Chatbot End-to-End

---

Take the **Section 6 Day 6 Enterprise RAG Chatbot** and put it on the internet with full CI/CD + observability + cost tracking + a public URL.

By the end of today you'll have a **shareable link** to a real deployed RAG app.


## Architecture

```
  git push
     │
     ▼
  ┌────────────────────┐
  │ GitHub Actions     │  test -> build -> push image to GHCR
  └────────┬───────────┘
           │
           ▼
  ┌────────────────────┐
  │ Render (or Fly)    │  auto-deploy on new image
  └────────┬───────────┘
           │
     public URL
           │
           ▼
  ┌────────────────────┐         ┌────────────────┐
  │ Users hit /ask     │────────►│   Langfuse     │  every call traced
  └────────┬───────────┘         └────────────────┘
           │
           ▼
  ┌────────────────────┐
  │ SQLite usage log   │  /metrics endpoint
  └────────────────────┘
```


## Step 1 — Add everything to your Section 6 app

Take `Section_06_RAG_Engineering/Day_6_Capstone_RAG_Chatbot/main.py` as your starting point. In one PR:

- Add `@lf.observe` decorators to `retrieve`, `rerank`, and `generate`
- Add the SQLite `usage` table from Day 5 (already exists in Section 6, just tag with `variant` too)
- Add `/metrics` and `/healthz` endpoints
- Add `/feedback` endpoint that posts to Langfuse `lf.score(...)`
- Add a `.dockerignore` and `Dockerfile` (Day 1)
- Add `.github/workflows/ci.yml` (Day 2)
- Add `render.yaml` (Day 3)


## Step 2 — Wire the secrets

**In GitHub:** Settings → Secrets and variables → Actions →

- `TOGETHER_API_KEY`
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `JWT_SECRET`

**In Render:** Environment tab → the same four.

Never commit `.env`.


## Step 3 — First deploy

```bash
git push origin main
```

Watch:
1. GitHub Actions run — tests → build → push image
2. Render pull image → boot → healthcheck green
3. You get a public URL like `https://my-rag.onrender.com/docs`

Fire a couple of `/ingest` and `/ask` calls. Confirm traces show up in Langfuse.


## Step 4 — Load-test it

Basic load test with `httpx`:


In [ ]:
import asyncio, httpx, time

async def hit(client, url, question):
    t0 = time.time()
    r = await client.post(url, json={"question": question}, timeout=30)
    return time.time() - t0, r.status_code

async def load_test(url, n=20, concurrency=5):
    async with httpx.AsyncClient() as client:
        sem = asyncio.Semaphore(concurrency)
        async def one():
            async with sem:
                return await hit(client, url, "What is 2+2?")
        results = await asyncio.gather(*[one() for _ in range(n)])

    lats = sorted(r[0] for r in results)
    print(f"n={n} concurrency={concurrency}")
    print(f"  p50 = {lats[n//2]:.2f}s")
    print(f"  p95 = {lats[int(n*0.95)-1]:.2f}s")
    print(f"  max = {lats[-1]:.2f}s")
    print(f"  errors = {sum(1 for r in results if r[1] != 200)}")

# asyncio.run(load_test("https://my-rag.onrender.com/ask", n=20, concurrency=5))
print("Fill in your Render URL and run.")


**Note down your p50 / p95.** This is your baseline. Every future perf change should be measured against it.


## Step 5 — Set an SLO

Pick two numbers you commit to:

- **Latency SLO** — e.g. "p95 < 4 s"
- **Availability SLO** — e.g. "99% of `/ask` responses succeed"

Configure your observability tool to alert when either breaks over a 30-minute window. Now you have a **production system with defined behavior**, not a demo.


## Deliverables checklist

Your capstone is done when you can share these:

- [ ] **Public URL** — anyone can hit `/docs`
- [ ] **GitHub repo URL** — CI green on `main`
- [ ] **Langfuse project link** with real traces (screenshots OK if project is private)
- [ ] **`/metrics` output** showing token + cost totals for the past day
- [ ] **Load-test numbers** — p50, p95, error rate
- [ ] **A stated SLO** on latency and availability
- [ ] **A one-line rollback plan** (usually: revert the merge, re-deploy)


## What to add next (portfolio polish)

Pick one:

1. **Auto-scaling** — Render's paid tier auto-scales replicas. Wire it up.
2. **Blue/green deploy** — deploy new version to a separate URL, canary 5% traffic, ramp up.
3. **Frontend** — one-page HTML that uses `/ask` with streaming. Deploy on Vercel (free).
4. **Public demo** — mirror the API to a Hugging Face Space with Gradio for a shareable UI.


## What you built in Section 9

Six days, one shippable production stack:

- **Docker** container for reproducible deploys (Day 1)
- **GitHub Actions** CI/CD — test, build, push, auto-deploy (Day 2)
- **Cloud PaaS** deploy — public URL with HTTPS (Day 3)
- **Langfuse** end-to-end tracing (Day 4)
- **SQLite** usage log + per-user budget caps (Day 5)
- **Load-tested** with numeric SLOs

You've now built and deployed **the exact stack behind most 2026 AI-startup MVPs.** Next up: Section 10 — system design & interview prep.
